# 04 — Усталость + минималистичная модель

**Блок A**: гипотеза усталости — скорость зависит от пройденного пути/времени/набора высоты.

**Блок B**: минималистичная модель — только `signed_slope_deg` + `difficulty`.

In [ ]:
import pickle, warnings
from pathlib import Path
from math import degrees, atan2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

CACHE_DIR  = Path("cache")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)
print("OK")


## Блок A — Гипотеза усталости

In [ ]:
df_raw = pickle.load(open(CACHE_DIR / "hikr_model_df.pkl", "rb"))
print(f"Сегментов: {len(df_raw):,}  треков: {df_raw['track_id'].nunique():,}")
print(df_raw.columns.tolist())


In [ ]:
df = df_raw.copy()

# Signed slope: + вверх, - вниз
df["signed_slope_deg"] = df.apply(
    lambda r: degrees(atan2(r["elev_diff_m"], max(r["dist_m"], 0.1))), axis=1
)

# Кумулятивные фичи внутри трека (сегменты уже в порядке из GPX)
# shift(1) - считаем то, что БЫЛО ДО текущего сегмента
df["cum_dist_m"]     = df.groupby("track_id")["dist_m"].cumsum().shift(1).fillna(0)
df["cum_time_s"]     = df.groupby("track_id")["dt_s"].cumsum().shift(1).fillna(0)
df["cum_elev_gain_m"] = (
    df.groupby("track_id")["elev_diff_m"]
    .transform(lambda x: x.clip(lower=0).cumsum().shift(1).fillna(0))
)

# Log-transform кумулятивных фичей (хвост длинный)
df["log_cum_dist"]  = np.log1p(df["cum_dist_m"])
df["log_cum_time"]  = np.log1p(df["cum_time_s"])
df["log_cum_gain"]  = np.log1p(df["cum_elev_gain_m"])

print("Кумулятивные фичи добавлены.")
print(df[["cum_dist_m","cum_time_s","cum_elev_gain_m"]].describe().round(1))


In [ ]:
# Быстрая проверка: корреляция кумулятивных фичей со скоростью
feats_check = ["cum_dist_m","cum_time_s","cum_elev_gain_m","signed_slope_deg","difficulty"]
corr = df[feats_check + ["speed_kmh"]].corr()["speed_kmh"].drop("speed_kmh")
print("Корреляция фичей с speed_kmh:")
print(corr.sort_values().round(3))

# График: скорость vs накопленная дистанция (биннинг)
bins = pd.cut(df["cum_dist_m"].clip(0, 30_000), bins=20)
mean_speed = df.groupby(bins)["speed_kmh"].mean()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
mean_speed.plot(ax=axes[0], marker="o", color="#3b82f6")
axes[0].set_title("Скорость vs накопленная дистанция"); axes[0].set_xlabel("cum_dist_m")

bins2 = pd.cut(df["cum_time_s"].clip(0, 14400), bins=20)
df.groupby(bins2)["speed_kmh"].mean().plot(ax=axes[1], marker="o", color="#f97316")
axes[1].set_title("Скорость vs накопленное время"); axes[1].set_xlabel("cum_time_s")

bins3 = pd.cut(df["cum_elev_gain_m"].clip(0, 3000), bins=20)
df.groupby(bins3)["speed_kmh"].mean().plot(ax=axes[2], marker="o", color="#22c55e")
axes[2].set_title("Скорость vs набор высоты"); axes[2].set_xlabel("cum_elev_gain_m")

plt.tight_layout(); plt.show()


In [ ]:
FEATURES_FATIGUE = [
    "signed_slope_deg", "log_cum_dist", "log_cum_time", "log_cum_gain", "difficulty"
]
CAT_FEATURES = ["difficulty"]
TARGET = "speed_kmh"

X = df[FEATURES_FATIGUE].copy()
y = df[TARGET].copy()
groups = df["track_id"].values

CB_PARAMS = dict(iterations=800, learning_rate=0.05, depth=6,
                 loss_function="MAE", eval_metric="MAE",
                 random_seed=42, verbose=0)

# GroupKFold CV
gkf = GroupKFold(n_splits=5)
cv_mae, cv_r2 = [], []
print("GroupKFold CV (усталость):")
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    m = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=CAT_FEATURES,
          eval_set=(X.iloc[val_idx], y.iloc[val_idx]), use_best_model=True)
    yp = m.predict(X.iloc[val_idx])
    cv_mae.append(mean_absolute_error(y.iloc[val_idx], yp))
    cv_r2.append(r2_score(y.iloc[val_idx], yp))
    print(f"  Fold {fold+1}: MAE={cv_mae[-1]:.3f}  R^2={cv_r2[-1]:.3f}")

print(f"\nCV: MAE={np.mean(cv_mae):.3f} +/- {np.std(cv_mae):.3f}  R^2={np.mean(cv_r2):.3f} +/- {np.std(cv_r2):.3f}")

# Финальная модель
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]

model_fat = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
model_fat.fit(X_train, y_train, cat_features=CAT_FEATURES,
              eval_set=(X_test, y_test), use_best_model=True)

yp_tr = model_fat.predict(X_train)
yp_te = model_fat.predict(X_test)
print(f"\nTrain: MAE={mean_absolute_error(y_train,yp_tr):.3f}  R^2={r2_score(y_train,yp_tr):.3f}")
print(f"Test:  MAE={mean_absolute_error(y_test,yp_te):.3f}  R^2={r2_score(y_test,yp_te):.3f}")

fi = pd.Series(model_fat.get_feature_importance(), index=FEATURES_FATIGUE).sort_values()
fi.plot(kind="barh", color="#3b82f6", figsize=(8, 4), title="Feature Importance (усталость)")
plt.tight_layout(); plt.show()


## Блок B — Минималистичная модель: только `signed_slope_deg` + `difficulty`

In [ ]:
FEATURES_MINI = ["signed_slope_deg", "difficulty"]
CAT_MINI      = ["difficulty"]

X_m = df[FEATURES_MINI].copy()
y_m = df[TARGET].copy()

gkf2 = GroupKFold(n_splits=5)
cv_mae_m, cv_r2_m = [], []
print("GroupKFold CV (мини-модель):")
for fold, (tr_idx, val_idx) in enumerate(gkf2.split(X_m, y_m, groups)):
    m = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
    m.fit(X_m.iloc[tr_idx], y_m.iloc[tr_idx], cat_features=CAT_MINI,
          eval_set=(X_m.iloc[val_idx], y_m.iloc[val_idx]), use_best_model=True)
    yp = m.predict(X_m.iloc[val_idx])
    cv_mae_m.append(mean_absolute_error(y_m.iloc[val_idx], yp))
    cv_r2_m.append(r2_score(y_m.iloc[val_idx], yp))
    print(f"  Fold {fold+1}: MAE={cv_mae_m[-1]:.3f}  R^2={cv_r2_m[-1]:.3f}")

print(f"\nCV: MAE={np.mean(cv_mae_m):.3f} +/- {np.std(cv_mae_m):.3f}  R^2={np.mean(cv_r2_m):.3f} +/- {np.std(cv_r2_m):.3f}")

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss2.split(X_m, y_m, groups=groups))
model_mini = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
model_mini.fit(X_m.iloc[tr_idx], y_m.iloc[tr_idx], cat_features=CAT_MINI,
               eval_set=(X_m.iloc[te_idx], y_m.iloc[te_idx]), use_best_model=True)

yp_te_m = model_mini.predict(X_m.iloc[te_idx])
print(f"Test:  MAE={mean_absolute_error(y_m.iloc[te_idx],yp_te_m):.3f}  R^2={r2_score(y_m.iloc[te_idx],yp_te_m):.3f}")


In [ ]:
print("=" * 55)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("=" * 55)
print(f"{'Модель':<30} {'CV R^2':>8}  {'CV MAE':>8}")
print("-" * 55)
print(f"{'Baseline Tobler':<30} {'-':>8}  {'1.186':>8}")
print(f"{'Notebook 03 (7 фичей)':<30} {'0.248':>8}  {'1.065':>8}")
print(f"{'Усталость (5 фичей)':<30} {np.mean(cv_r2):>8.3f}  {np.mean(cv_mae):>8.3f}")
print(f"{'Мини (slope+difficulty)':<30} {np.mean(cv_r2_m):>8.3f}  {np.mean(cv_mae_m):>8.3f}")
print("=" * 55)
